# Mock Exam: Model Too Bad - Performance Issues (Interactive Version)

## 🎯 Your Mission
You've trained a model with only **55% accuracy** - barely better than random guessing!

**Your task:**
1. Run the code below and observe the terrible results
2. Find ALL issues causing poor performance (there are 10+ major bugs)
3. Fix them step by step
4. Achieve realistic performance (~80-85% accuracy)

## 📝 Instructions
1. First, generate the dataset by running the "Setup" cell
2. Then run the "Bad Model" cells and observe the results
3. Try to identify what's wrong before looking at hints
4. Fix the issues in the "Your Fixed Version" section

**Time limit:** 60 minutes  
**Categories of bugs:** Missing values, outliers, feature engineering, encoding, scaling, class imbalance, model selection

## Setup: Generate Dataset

In [37]:
# Run this first to generate the dataset
%run generate_dataset.py

Generating dataset for 'model too bad' exam...
Original dataset shape: (20640, 9)
Target distribution:
expensive
0    14858
1     5782
Name: count, dtype: int64

✅ Dataset saved to: housing_for_bad_model.csv
Final shape: (20640, 12)

Columns: ['median_income', 'housing_median_age', 'avg_rooms', 'avg_bedrooms', 'population', 'avg_occupancy', 'latitude', 'longitude', 'ocean_proximity', 'neighborhood', 'block_id', 'expensive']

Missing values:
median_income         2024
housing_median_age       0
avg_rooms                0
avg_bedrooms          3062
population               0
avg_occupancy            0
latitude                 0
longitude                0
ocean_proximity       4249
neighborhood             0
block_id                 0
expensive                0
dtype: int64

⚠️ This dataset has several issues that will cause poor model performance!
Issues include: missing values, outliers, high-cardinality categorical, class imbalance, etc.

✅ Dataset saved to: housing_for_bad_model.csv
F

## Part 1: The Bad Model (Run and Observe)

In [38]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Load data
print("Loading data...")
df = pd.read_csv("housing_for_bad_model.csv")
print(f"Dataset shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nTarget distribution:\n{df['expensive'].value_counts()}")
df.head()

Loading data...
Dataset shape: (20640, 12)

Missing values:
median_income         2024
housing_median_age       0
avg_rooms                0
avg_bedrooms          3062
population               0
avg_occupancy            0
latitude                 0
longitude                0
ocean_proximity       4249
neighborhood             0
block_id                 0
expensive                0
dtype: int64

Target distribution:
expensive
0    14858
1     5782
Name: count, dtype: int64


,median_income,housing_median_age,avg_rooms,avg_bedrooms,population,avg_occupancy,latitude,longitude,ocean_proximity,neighborhood,block_id,expensive
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,NEAR OCEAN,Urban,BLOCK_0,1
1,8.3014,21.0,6.238137,NaN,2401.0,2.109842,37.86,-122.22,INLAND,Urban,BLOCK_1,1
2,NaN,52.0,8.288136,NaN,496.0,2.802260,37.85,-122.24,INLAND,Urban,BLOCK_2,1
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,<1H OCEAN,Urban,BLOCK_3,1
4,3.8462,52.0,6.281853,1.081081,5650.0,2.181467,37.85,-122.25,NEAR BAY,Urban,BLOCK_4,1


### 🐛 BUG ZONE #1: Missing Values

**Question:** How should we handle missing values?

In [39]:
# BUG: Dropping ALL rows with ANY missing value!
print(f"Shape before dropping: {df.shape}")
df_clean = df.dropna()  # This removes ~30% of data!
print(f"Shape after dropping: {df_clean.shape}")
print(f"Lost {len(df) - len(df_clean)} rows ({100*(len(df) - len(df_clean))/len(df):.1f}%)")

# YOUR ANSWER: What's wrong with this approach?
# Answer: _______________________________

Shape before dropping: (20640, 12)
Shape after dropping: (12567, 12)
Lost 8073 rows (39.1%)


### 🐛 BUG ZONE #2: Categorical Encoding

**Question:** Is this the right way to encode categorical variables?

In [40]:
# BUG #1: Using Label Encoding for non-ordinal categorical variables!
le_ocean = LabelEncoder()
le_neighborhood = LabelEncoder()
le_block = LabelEncoder()

df_clean['ocean_proximity_encoded'] = le_ocean.fit_transform(df_clean['ocean_proximity'])
df_clean['neighborhood_encoded'] = le_neighborhood.fit_transform(df_clean['neighborhood'])

# BUG #2: Encoding high-cardinality categorical as integer!
df_clean['block_id_encoded'] = le_block.fit_transform(df_clean['block_id'])

# BUG #3: Adding TONS of random noise as features (destroys signal!)
np.random.seed(42)
for i in range(100):  # Add 100 random noise columns!
    df_clean[f'random_noise_{i}'] = np.random.randn(len(df_clean))

# BUG #4: DROP the most important features!
df_clean = df_clean.drop(['median_income', 'housing_median_age'], axis=1)

print("Unique values in block_id:", df_clean['block_id'].nunique())
print(f"Added 100 random noise features!")
print(f"Dropped median_income and housing_median_age (most predictive features)!")
print("\n⚠️ This creates ordinal relationships where none exist!")
print("⚠️ Also, 5000 unique blocks will create problems!")
print("⚠️ Random noise features will confuse the model!")
print("⚠️ Dropped the best predictive features!")

# Drop original categorical columns
df_clean = df_clean.drop(['ocean_proximity', 'neighborhood', 'block_id'], axis=1)

# YOUR ANSWER: What's wrong here?
# Problem 1: _______________________________
# Problem 2: _______________________________
# Problem 3: _______________________________
# Problem 4: _______________________________

Unique values in block_id: 4905
Added 100 random noise features!
Dropped median_income and housing_median_age (most predictive features)!

⚠️ This creates ordinal relationships where none exist!
⚠️ Also, 5000 unique blocks will create problems!
⚠️ Random noise features will confuse the model!
⚠️ Dropped the best predictive features!


### 🐛 BUG ZONE #3: Outliers and Scaling

**Question:** Should we check for outliers before scaling?

In [41]:
# BUG: No outlier detection/treatment!
print("Population statistics:")
print(df_clean['population'].describe())
print(f"\nMax population: {df_clean['population'].max():.0f}")
print(f"99th percentile: {df_clean['population'].quantile(0.99):.0f}")
print("\n⚠️ Notice the extreme outliers!")

# BUG: Scaling with outliers present - ruins the scaling!
X = df_clean.drop('expensive', axis=1)
y = df_clean['expensive']

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)

print("\nScaled data - most values squeezed into tiny range:")
print(X_scaled.describe())

# YOUR ANSWER: How do outliers affect StandardScaler?
# Answer: _______________________________

Population statistics:
count    12567.000000
mean      1460.986393
std       1406.106721
min          3.000000
25%        791.000000
50%       1167.000000
75%       1733.500000
max      56960.000000
Name: population, dtype: float64

Max population: 56960
99th percentile: 6376

⚠️ Notice the extreme outliers!

Scaled data - most values squeezed into tiny range:
          avg_rooms  avg_bedrooms    population  avg_occupancy      latitude  \
count  1.256700e+04  1.256700e+04  1.256700e+04   1.256700e+04  1.256700e+04   
mean   3.143644e-16  3.957825e-17  7.915651e-18  -1.470049e-17  2.533008e-16   
std    1.000040e+00  1.000040e+00  1.000040e+00   1.000040e+00  1.000040e+00   
min   -1.896528e+00 -1.562474e+00 -1.036937e+00  -1.978048e-01 -1.453480e+00   
25%   -4.072180e-01 -1.860034e-01 -4.765023e-01  -5.505572e-02 -8.035437e-01   
50%   -8.107892e-02 -9.984823e-02 -2.090866e-01  -2.295060e-02 -6.492421e-01   
75%    2.613496e-01  5.650328e-03  1.938149e-01   1.512122e-02  9.732619e-01 

### 🐛 BUG ZONE #4: Feature Engineering

**Question:** Are we creating useful features?

In [42]:
# BUG: No feature engineering at all!
# Missing obvious interactions and ratios

print("Current features:")
print(X_scaled.columns.tolist())
print("\n⚠️ No domain-specific features created!")
print("⚠️ Latitude/Longitude used as-is (should create location features)")
print("⚠️ No room/population ratios")

# YOUR ANSWER: What features would be useful?
# Feature 1: _______________________________
# Feature 2: _______________________________
# Feature 3: _______________________________

Current features:
['avg_rooms', 'avg_bedrooms', 'population', 'avg_occupancy', 'latitude', 'longitude', 'ocean_proximity_encoded', 'neighborhood_encoded', 'block_id_encoded', 'random_noise_0', 'random_noise_1', 'random_noise_2', 'random_noise_3', 'random_noise_4', 'random_noise_5', 'random_noise_6', 'random_noise_7', 'random_noise_8', 'random_noise_9', 'random_noise_10', 'random_noise_11', 'random_noise_12', 'random_noise_13', 'random_noise_14', 'random_noise_15', 'random_noise_16', 'random_noise_17', 'random_noise_18', 'random_noise_19', 'random_noise_20', 'random_noise_21', 'random_noise_22', 'random_noise_23', 'random_noise_24', 'random_noise_25', 'random_noise_26', 'random_noise_27', 'random_noise_28', 'random_noise_29', 'random_noise_30', 'random_noise_31', 'random_noise_32', 'random_noise_33', 'random_noise_34', 'random_noise_35', 'random_noise_36', 'random_noise_37', 'random_noise_38', 'random_noise_39', 'random_noise_40', 'random_noise_41', 'random_noise_42', 'random_noise_43',

### 🐛 BUG ZONE #5: Train-Test Split

**Question:** Is this split appropriate?

In [43]:
# BUG: Not using stratified split with imbalanced classes!
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
    # Missing: stratify=y
)

print("Training set target distribution:")
print(y_train.value_counts(normalize=True))
print("\nTest set target distribution:")
print(y_test.value_counts(normalize=True))
print("\n⚠️ Notice the class imbalance!")

# YOUR ANSWER: Why is stratification important here?
# Answer: _______________________________

Training set target distribution:
expensive
0    0.720283
1    0.279717
Name: proportion, dtype: float64

Test set target distribution:
expensive
0    0.727924
1    0.272076
Name: proportion, dtype: float64

⚠️ Notice the class imbalance!


### 🐛 BUG ZONE #6: Model Selection and Training

**Question:** Is Logistic Regression with default settings appropriate?

In [44]:
# BUG: Using default LogisticRegression with imbalanced data!
# BUG: No hyperparameter tuning
# BUG: No handling of class imbalance
# BUG: Using LINEAR model when data likely has non-linear relationships!
# BUG: Setting terrible hyperparameters!

model = LogisticRegression(
    random_state=42,
    max_iter=5,      # BUG: Way too few iterations!
    C=0.00001,       # BUG: Extremely high regularization (kills model)!
    penalty='l2'     # With tiny C, this over-regularizes
)
# Missing: class_weight='balanced'
# Missing: max_iter tuning (should be 1000+)
# Missing: regularization tuning (C should be ~1.0)
# Should use: RandomForest or other non-linear model

model.fit(X_train, y_train)

print("Model trained with TERRIBLE parameters:")
print(f"  - max_iter=5 (won't converge)")
print(f"  - C=0.00001 (extreme regularization)")
print(f"  - No class balancing")
print(f"\nClasses in training: {np.unique(y_train)}")
print(f"Class distribution: {y_train.value_counts().to_dict()}")

# YOUR ANSWER: What parameters should be adjusted?
# Parameter 1: _______________________________
# Parameter 2: _______________________________
# Parameter 3: _______________________________
# Parameter 4: _______________________________

Model trained with TERRIBLE parameters:
  - max_iter=5 (won't converge)
  - C=0.00001 (extreme regularization)
  - No class balancing

Classes in training: [0 1]
Class distribution: {0: 7241, 1: 2812}


### Evaluate the Bad Model

In [45]:
# Make predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print("\n" + "="*60)
print("😢 RESULTS (TERRIBLE PERFORMANCE!) 😢")
print("="*60)
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

print(f"\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred))

print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\n⚠️ Notice the poor performance!")
print("⚠️ Check precision/recall for each class")
print("⚠️ Model might be predicting mostly one class")


😢 RESULTS (TERRIBLE PERFORMANCE!) 😢
Training Accuracy: 0.7203
Test Accuracy: 0.7279

Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.73      1.00      0.84      1830
           1       0.00      0.00      0.00       684

    accuracy                           0.73      2514
   macro avg       0.36      0.50      0.42      2514
weighted avg       0.53      0.73      0.61      2514


Confusion Matrix:
[[1830    0]
 [ 684    0]]

⚠️ Notice the poor performance!
⚠️ Check precision/recall for each class
⚠️ Model might be predicting mostly one class


### Analysis: What Went Wrong?

In [ ]:
# Check prediction distribution
print("Prediction distribution (test set):")
print(pd.Series(y_test_pred).value_counts())

print("\nActual distribution (test set):")
print(y_test.value_counts())

print("\n⚠️ Is the model biased toward one class?")

## Part 2: Your Turn - Fix the Issues!

### 📝 Before you start coding:

**List all the bugs you found:**

1. Missing values: _______________________________
2. Categorical encoding: _______________________________
3. High cardinality: _______________________________
4. Outliers: _______________________________
5. Scaling: _______________________________
6. Feature engineering: _______________________________
7. Train-test split: _______________________________
8. Class imbalance: _______________________________
9. Model selection: _______________________________
10. Hyperparameters: _______________________________

### Now implement your fixes below:

In [ ]:
# YOUR FIXED VERSION HERE
# Start fresh - reload the data

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

np.random.seed(42)

# Load data
df = pd.read_csv("housing_for_bad_model.csv")

# TODO: Fix #1 - Handle missing values properly
# Your code here:


# TODO: Fix #2 - Handle outliers
# Your code here:


# TODO: Fix #3 - Create useful features
# Your code here:


# TODO: Fix #4 - Prepare X and y (remove target, handle high-cardinality)
# Your code here:


# TODO: Fix #5 - Split with stratification
# Your code here:


# TODO: Fix #6 - Build proper pipeline with correct encoding and scaling
# Your code here:


# TODO: Fix #7 - Train model with class_weight and tuned hyperparameters
# Your code here:


# TODO: Fix #8 - Evaluate properly
# Your code here:

## Part 3: Verify Your Fixes

After implementing your fixes, answer these questions:

1. **What is your new test accuracy?** _______  
   (Should be around 80-85%)

2. **How does it compare to the bad model?**  
   _______________________________

3. **Are both classes being predicted well?**  
   _______________________________

4. **Which fix had the biggest impact?**  
   _______________________________

5. **What features are now most important?**  
   _______________________________

## Part 4: Reflection

### Key Takeaways

Write down the most important lessons you learned:

1. _______________________________
2. _______________________________
3. _______________________________
4. _______________________________
5. _______________________________

### Real-World Application

How will you ensure good model performance in your future projects?

_______________________________
_______________________________
_______________________________

---

## 🎓 Common Mistakes Summary

**Data Quality Issues:**
- Dropping too much data instead of imputing
- Ignoring outliers
- Not checking data distributions

**Feature Engineering:**
- Using wrong encoding for categorical variables
- Not handling high-cardinality categoricals
- Missing domain-specific features
- Not creating interaction features

**Preprocessing:**
- Using wrong scaler (StandardScaler with outliers)
- Not using pipelines (causes data leakage)
- Scaling before splitting

**Modeling:**
- Ignoring class imbalance
- Not tuning hyperparameters
- Using inappropriate model for the data
- Not using stratified split

**Evaluation:**
- Only looking at accuracy (misleading with imbalanced data)
- Not checking per-class metrics
- Not using cross-validation